In [16]:
import os
import pandas as pd
import numpy as np

In [17]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "trivia_qa": "TriviaQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

In [18]:
def train_test_dataset_split(combined_name: str):
    # Find positions of known datasets in the string
    positions = []
    for key in dataset_map.keys():
        idx = combined_name.find(key)
        if idx != -1:
            positions.append((idx, key))
    
    if len(positions) == 2:
        # Sort by occurrence in string
        positions.sort(key=lambda x: x[0])
        train_set = dataset_map[positions[0][1]]
        test_set = dataset_map[positions[1][1]]
        return train_set, test_set
    
    raise ValueError(f"Unrecognized dataset name in: {combined_name}")


# Tests
print(train_test_dataset_split("trivia_qa_mmlu"))  # ('TriviaQA', 'MMLU')
print(train_test_dataset_split("mmlu_trivia_qa"))  # ('MMLU', 'TriviaQA')

('TriviaQA', 'MMLU')
('MMLU', 'TriviaQA')


In [19]:
results_dir = f"/hdd/ivny/cross_domain_results"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, estimation_method, _, model_name, train_test_sets = leaf_dir.split("/")
    train_set, test_set = train_test_dataset_split(train_test_sets)
    csv_data = pd.read_csv(os.path.join(leaf_dir, "linguistic_calibration_metrics.csv")).set_index("metric").transpose().to_dict(orient="records")[0]
    df_dict = {
        "Train": train_set,
        "Test": test_set,
        "Est. Method": methods_names[estimation_method],
        "Model": model_name_map[model_name],
        **csv_data
    }
    all_records.append(df_dict)

full_df = pd.DataFrame(all_records)

In [20]:
full_df.head(10)

,Train,Test,Est. Method,Model,original_signal_generalised_ECE,original_signal_dECE,original_signal_dECE_pt,original_signal_dAUROC,original_signal_auroc_pt,calibrated_signal_generalised_ECE,...,original_linguistic_generalised_ECE,original_linguistic_dECE,original_linguistic_dECE_pt,original_linguistic_dAUROC,original_linguistic_auroc_pt,calibrated_linguistic_generalised_ECE,calibrated_linguistic_dECE,calibrated_linguistic_dECE_pt,calibrated_linguistic_dAUROC,calibrated_linguistic_auroc_pt
0,SQuAD2.0,TriviaQA,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.061371,0.065341,0.049511,0.745909,0.792721,0.212097,...,0.085407,0.088967,0.049047,0.737110,0.787483,0.173153,0.176770,0.167389,0.688619,0.742747
1,MMLU,SQuAD2.0,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.247557,0.247930,0.231192,0.622322,0.646582,0.154349,...,0.253626,0.253886,0.227704,0.617697,0.645463,0.199142,0.199371,0.169205,0.596804,0.619813
2,TriviaQA,SQuAD2.0,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.247700,0.247992,0.231192,0.621055,0.646582,0.263924,...,0.254566,0.255107,0.229408,0.615538,0.644511,0.288161,0.288473,0.258132,0.598727,0.615429
3,TriviaQA,MMLU,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.123723,0.126150,0.093252,0.608879,0.645463,0.145255,...,0.147068,0.148691,0.087825,0.602600,0.640816,0.182435,0.183699,0.142350,0.606179,0.635800
4,MMLU,TriviaQA,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.061405,0.065324,0.049511,0.745964,0.792721,0.104421,...,0.083861,0.087670,0.047792,0.738754,0.789122,0.105683,0.111156,0.082622,0.720816,0.764055
5,SQuAD2.0,MMLU,Dist. Ling. Conf.,Llama-3.1-8B-Inst.,0.123737,0.126116,0.093252,0.608266,0.645463,0.136651,...,0.147943,0.149532,0.087158,0.601328,0.640077,0.138630,0.140590,0.087302,0.587898,0.620159
6,SQuAD2.0,TriviaQA,Dist. Ling. Conf.,Llama-3-8B-Inst.,0.098022,0.099472,0.053152,0.628552,0.666004,0.226991,...,0.132611,0.133754,0.066988,0.615159,0.655024,0.219348,0.220738,0.201529,0.581926,0.627752
7,MMLU,SQuAD2.0,Dist. Ling. Conf.,Llama-3-8B-Inst.,0.306121,0.306733,0.291594,0.547334,0.568762,0.143151,...,0.314012,0.314682,0.291756,0.544928,0.566224,0.194341,0.194625,0.157426,0.514318,0.521500
8,TriviaQA,SQuAD2.0,Dist. Ling. Conf.,Llama-3-8B-Inst.,0.306283,0.306730,0.291594,0.546897,0.568762,0.263625,...,0.312909,0.313665,0.288872,0.546021,0.567288,0.317991,0.318722,0.291489,0.537304,0.547624
9,TriviaQA,MMLU,Dist. Ling. Conf.,Llama-3-8B-Inst.,0.190040,0.191331,0.165915,0.557960,0.582978,0.157575,...,0.213267,0.213531,0.158590,0.550928,0.580533,0.218228,0.218622,0.184146,0.549716,0.574468


In [21]:
df = full_df.copy()

# All datasets
datasets = sorted(set(df["Train"]).union(set(df["Test"])))

def build_table(df, value_col):
    """
    Build directional Train × Test matrix.
    Averages across models.
    Diagonal entries forced to NaN.
    """
    # Average across models for each directional pair
    grouped = (
        df.groupby(["Train", "Test"])[value_col]
        .mean()
    )
    
    # Create full empty matrix
    table = pd.DataFrame(index=datasets, columns=datasets, dtype=float)
    
    # Fill directional values
    for (train, test), value in grouped.items():
        table.loc[train, test] = value
    
    # Remove diagonal (no self train-test)
    for d in datasets:
        table.loc[d, d] = np.nan
        
    return table

# Build difference tables for all methods
method_tables = {}
for method in methods_names.values():
    df_method = df[df["Est. Method"] == method].copy()
    original_table = build_table(df_method, "original_linguistic_dECE")
    calibrated_table = build_table(df_method, "calibrated_linguistic_dECE")
    difference_table = calibrated_table - original_table
    method_tables[method] = difference_table

# Build combined LaTeX table
latex_lines = []
latex_lines.append(r"\begin{table}[h!]")
latex_lines.append(r"\centering")
latex_lines.append(r"\setlength{\tabcolsep}{6pt}")
latex_lines.append(r"\small")
latex_lines.append(r"\caption{\textbf{Cross-Domain Post-hoc Linguistic Calibration dECE Improvement.} The results shows the mean dECE improvement (reduction) after applying post-hoc linguistic calibration, grouped by estimation method. Each entry represents the average improvement when training on the dataset in the row and testing on the dataset in the column, averaged across all models. Diagonal entries are not applicable since they represent in-domain calibration.}")
latex_lines.append(r"\label{tab:cross-domain-linguistic-calibration-dECE-improvement}")
latex_lines.append(r"\begin{tabular}{llrrr}")
latex_lines.append(r"\toprule")

# Header with Train/Test labels
header = r"Est.\ Method & Train/Test & " + " & ".join(datasets) + r" \\"
latex_lines.append(header)
latex_lines.append(r"\midrule")
latex_lines.append("")

# Add each method's rows
for i, (method, table) in enumerate(method_tables.items()):
    # Escape special characters in method name
    method_escaped = method.replace(".", ".\\ ")
    
    latex_lines.append(r"\multirow{3}{*}{" + method_escaped + "}")
    
    for j, train_dataset in enumerate(datasets):
        if j == 0:
            row_start = ""
        else:
            row_start = ""
        
        values = []
        for test_dataset in datasets:
            val = table.loc[train_dataset, test_dataset]
            if pd.isna(val):
                values.append("-")
            else:
                values.append(f"{val:.3f}")
        
        row = row_start + f"& {train_dataset:9s} & " + " & ".join(f"{v:>5s}" for v in values) + r" \\"
        latex_lines.append(row)
    
    # Add midrule after each method except the last
    if i < len(method_tables) - 1:
        latex_lines.append(r"\midrule")
        latex_lines.append("")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\end{table}")

combined_latex = "\n".join(latex_lines)
print(combined_latex)

\begin{table}[h!]
\centering
\setlength{\tabcolsep}{6pt}
\small
\caption{\textbf{Cross-Domain Post-hoc Linguistic Calibration dECE Improvement.} The results shows the mean dECE improvement (reduction) after applying post-hoc linguistic calibration, grouped by estimation method. Each entry represents the average improvement when training on the dataset in the row and testing on the dataset in the column, averaged across all models. Diagonal entries are not applicable since they represent in-domain calibration.}
\label{tab:cross-domain-linguistic-calibration-dECE-improvement}
\begin{tabular}{llrrr}
\toprule
Est.\ Method & Train/Test & MMLU & SQuAD2.0 & TriviaQA \\
\midrule

\multirow{3}{*}{Dist.\  Ling.\  Conf.\ }
& MMLU      &     - & -0.070 & -0.005 \\
& SQuAD2.0  & 0.008 &     - & 0.009 \\
& TriviaQA  & 0.008 & -0.054 &     - \\
\midrule

\multirow{3}{*}{Dist.\  Semantic Unc.\ }
& MMLU      &     - & -0.186 & -0.047 \\
& SQuAD2.0  & -0.249 &     - & -0.047 \\
& TriviaQA  & -0.123 & -0

In [22]:
df = full_df.copy()

# All datasets
datasets = sorted(set(df["Train"]).union(set(df["Test"])))

def build_table(df, value_col):
    """
    Build directional Train × Test matrix.
    Averages across models.
    Diagonal entries forced to NaN.
    """
    # Average across models for each directional pair
    grouped = (
        df.groupby(["Train", "Test"])[value_col]
        .mean()
    )
    
    # Create full empty matrix
    table = pd.DataFrame(index=datasets, columns=datasets, dtype=float)
    
    # Fill directional values
    for (train, test), value in grouped.items():
        table.loc[train, test] = value
    
    # Remove diagonal (no self train-test)
    for d in datasets:
        table.loc[d, d] = np.nan
        
    return table

# Build difference tables for all methods
method_tables = {}
for method in methods_names.values():
    df_method = df[df["Est. Method"] == method].copy()
    original_table = build_table(df_method, "original_signal_dECE")
    calibrated_table = build_table(df_method, "calibrated_signal_dECE")
    difference_table = calibrated_table - original_table
    method_tables[method] = difference_table

# Build combined LaTeX table
latex_lines = []
latex_lines.append(r"\begin{table}[h!]")
latex_lines.append(r"\centering")
latex_lines.append(r"\setlength{\tabcolsep}{6pt}")
latex_lines.append(r"\small")
latex_lines.append(r"\caption{\textbf{Cross-Domain Post-hoc Signal Space Calibration dECE Improvement.} The results shows the mean dECE improvement (reduction) after applying post-hoc numerical calibration in the signal space, grouped by estimation method. Each entry represents the average improvement when training on the dataset in the row and testing on the dataset in the column, averaged across all models. Diagonal entries are not applicable since they represent in-domain calibration.}")
latex_lines.append(r"\label{tab:cross-domain-signal-calibration-dECE-improvement}")
latex_lines.append(r"\begin{tabular}{llrrr}")
latex_lines.append(r"\toprule")

# Header with Train/Test labels
header = r"Est.\ Method & Train/Test & " + " & ".join(datasets) + r" \\"
latex_lines.append(header)
latex_lines.append(r"\midrule")
latex_lines.append("")

# Add each method's rows
for i, (method, table) in enumerate(method_tables.items()):
    # Escape special characters in method name
    method_escaped = method.replace(".", ".\\ ")
    
    latex_lines.append(r"\multirow{3}{*}{" + method_escaped + "}")
    
    for j, train_dataset in enumerate(datasets):
        if j == 0:
            row_start = ""
        else:
            row_start = ""
        
        values = []
        for test_dataset in datasets:
            val = table.loc[train_dataset, test_dataset]
            if pd.isna(val):
                values.append("-")
            else:
                values.append(f"{val:.3f}")
        
        row = row_start + f"& {train_dataset:9s} & " + " & ".join(f"{v:>5s}" for v in values) + r" \\"
        latex_lines.append(row)
    
    # Add midrule after each method except the last
    if i < len(method_tables) - 1:
        latex_lines.append(r"\midrule")
        latex_lines.append("")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\end{table}")

combined_latex = "\n".join(latex_lines)
print(combined_latex)

\begin{table}[h!]
\centering
\setlength{\tabcolsep}{6pt}
\small
\caption{\textbf{Cross-Domain Post-hoc Signal Space Calibration dECE Improvement.} The results shows the mean dECE improvement (reduction) after applying post-hoc numerical calibration in the signal space, grouped by estimation method. Each entry represents the average improvement when training on the dataset in the row and testing on the dataset in the column, averaged across all models. Diagonal entries are not applicable since they represent in-domain calibration.}
\label{tab:cross-domain-signal-calibration-dECE-improvement}
\begin{tabular}{llrrr}
\toprule
Est.\ Method & Train/Test & MMLU & SQuAD2.0 & TriviaQA \\
\midrule

\multirow{3}{*}{Dist.\  Ling.\  Conf.\ }
& MMLU      &     - & -0.127 & -0.012 \\
& SQuAD2.0  & 0.022 &     - & 0.030 \\
& TriviaQA  & 0.017 & -0.091 &     - \\
\midrule

\multirow{3}{*}{Dist.\  Semantic Unc.\ }
& MMLU      &     - & -0.106 & -0.100 \\
& SQuAD2.0  & -0.179 &     - & -0.085 \\
& Trivia